# Berry Picker Assistant – Berry Detection (YOLO)
**Team Picky · ML team · TIES4570 CSDP**

This notebook does everything from scratch and produces a **handoff package** for the dev team:

1. Download WildBe (Hugging Face) → 2. Convert to YOLO format → 3. Train YOLO11n on GPU → 4. Test on unseen images → 5. Export to ONNX → 6. Package `best.onnx`, `predict.py`, metrics and README into `handoff.zip`

**Settings needed (right panel):** Accelerator = **GPU T4** · Internet = **On**. Then click **Save Version → Save & Run All (Commit)** and close the browser; it keeps running.

Dataset: WildBe (Riz et al., ECCV 2024 Workshops), licence **CC BY-NC 4.0** – non-commercial use only.

In [ ]:
!nvidia-smi

## 1. Install tools

In [ ]:
!pip install -q ultralytics onnx onnxslim onnxruntime huggingface_hub pyarrow

## 2. Download WildBe

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="FBK-TeV/WildBe", repo_type="dataset", local_dir="/tmp/wildbe")
!ls -la /tmp/wildbe/data

## 3. Convert to YOLO format (train 90% / test 10% by source photo, val = official validation)

In [ ]:
%%writefile /kaggle/working/prepare_wildbe.py
"""
prepare_wildbe.py  -  Team Picky / Berry Picker Assistant (ML team)

Turns the WildBe .parquet files into the folder layout YOLO needs:

  ML/data/wildbe_yolo/
      images/train  images/val  images/test
      labels/train  labels/val  labels/test
      data.yaml

- train.parquet      -> split into train (90%) and test (10%)
                        (split by source image, so crops of the same photo
                         never end up in both train and test)
- validation.parquet -> val

Run from:  C:\\Users\\Sayeem09\\Downloads\\TIES4570_CSDP_Sayeem
Command :  python ML\\prepare_wildbe.py
"""
import json
import random
import sys
from collections import Counter
from pathlib import Path

import pyarrow.parquet as pq

CLASSES = {0: "bilberry", 1: "cloudberry", 2: "crowberry", 3: "lingonberry"}
TEST_SHARE = 0.10
SEED = 42

src = Path(sys.argv[1]) if len(sys.argv) > 1 else Path("ML/data/wildbe/data")
out = Path(sys.argv[2]) if len(sys.argv) > 2 else Path("ML/data/wildbe_yolo")


def write_example(row, split, counts, boxes):
    name = f"wildbe_{int(row['index']):05d}"
    img = row["image"]["bytes"]
    ext = ".png" if img[:4] == b"\x89PNG" else ".jpg"
    (out / "images" / split / f"{name}{ext}").write_bytes(img)
    lines = []
    for lab in json.loads(row["labels"] or "[]"):
        c = int(lab["class"])
        lines.append(f"{c} {lab['x']:.6f} {lab['y']:.6f} {lab['width']:.6f} {lab['height']:.6f}")
        boxes[split][CLASSES[c]] += 1
    (out / "labels" / split / f"{name}.txt").write_text("\n".join(lines))
    counts[split] += 1


def main():
    for part in ("images", "labels"):
        for split in ("train", "val", "test"):
            (out / part / split).mkdir(parents=True, exist_ok=True)

    train_file = src / "train.parquet"
    val_file = src / "validation.parquet"
    for f in (train_file, val_file):
        if not f.exists():
            sys.exit(f"ERROR: cannot find {f}. Run this from the TIES4570_CSDP_Sayeem folder.")

    # 1) Decide which source photos go to test (no image data loaded yet)
    ids = pq.read_table(train_file, columns=["source_image_id"]).column(0).to_pylist()
    groups = sorted({str(i) for i in ids})
    random.Random(SEED).shuffle(groups)
    test_groups = set(groups[: max(1, int(len(groups) * TEST_SHARE))])

    counts, boxes = Counter(), {s: Counter() for s in ("train", "val", "test")}
    cols = ["index", "image", "labels", "source_image_id"]

    # 2) Write files in small batches so the PC does not run out of memory
    print("Unpacking train.parquet (this takes a few minutes)...")
    for batch in pq.ParquetFile(train_file).iter_batches(batch_size=100, columns=cols):
        for row in batch.to_pylist():
            split = "test" if str(row["source_image_id"]) in test_groups else "train"
            write_example(row, split, counts, boxes)
        print(f"  {counts['train'] + counts['test']} images done", end="\r")

    print("\nUnpacking validation.parquet...")
    for batch in pq.ParquetFile(val_file).iter_batches(batch_size=100, columns=cols):
        for row in batch.to_pylist():
            write_example(row, "val", counts, boxes)

    yaml = (
        f"path: {out.resolve().as_posix()}\n"
        "train: images/train\nval: images/val\ntest: images/test\n"
        "names:\n" + "".join(f"  {k}: {v}\n" for k, v in CLASSES.items())
    )
    (out / "data.yaml").write_text(yaml)

    print("\nDONE")
    for split in ("train", "val", "test"):
        b = boxes[split]
        print(f"  {split:5}: {counts[split]:5} images | "
              + "  ".join(f"{CLASSES[c]} {b[CLASSES[c]]}" for c in CLASSES))
    print(f"  data.yaml written to {out / 'data.yaml'}")


if __name__ == "__main__":
    main()


In [ ]:
!python /kaggle/working/prepare_wildbe.py /tmp/wildbe/data /tmp/wildbe_yolo
!cat /tmp/wildbe_yolo/data.yaml

## 4. Train YOLO11n (small + fast → right size for a mobile/backend service)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(
    data="/tmp/wildbe_yolo/data.yaml",
    epochs=100, patience=20, imgsz=640, batch=32,
    device=0, workers=4, seed=42,
    project="/kaggle/working/runs", name="wildbe_v1", exist_ok=True,
)

## 5. Test on images the model has never seen

In [ ]:
import json
best = YOLO("/kaggle/working/runs/wildbe_v1/weights/best.pt")
m = best.val(data="/tmp/wildbe_yolo/data.yaml", split="test", imgsz=640, device=0,
             project="/kaggle/working/runs", name="wildbe_v1_test", exist_ok=True)
names = ["bilberry", "cloudberry", "crowberry", "lingonberry"]
metrics = {
    "test_mAP50": round(float(m.box.map50), 3),
    "test_mAP50_95": round(float(m.box.map), 3),
    "precision": round(float(m.box.mp), 3),
    "recall": round(float(m.box.mr), 3),
    "per_class_mAP50": {names[i]: round(float(v), 3) for i, v in zip(m.box.ap_class_index, m.box.ap50)},
}
print(json.dumps(metrics, indent=2))

## 6. Export to ONNX (for the FastAPI backend)

In [ ]:
onnx_path = best.export(format="onnx", imgsz=640, simplify=True)
print(onnx_path)

## 7. Build the handoff package for the dev team

In [ ]:
%%writefile /kaggle/working/predict.py
"""
predict.py  -  Berry detector for Berry Picker Assistant (handoff to the dev team)

Usage (FastAPI backend):
    from predict import BerryDetector
    detector = BerryDetector("best.onnx")          # load once at startup
    result = detector.detect("photo.jpg")           # path, bytes, PIL image or numpy array

Returns:
    {
      "berries": [ {"berry": "bilberry", "confidence": 0.87, "box": [x1, y1, x2, y2]}, ... ],
      "counts":  {"bilberry": 12, "cloudberry": 0, "crowberry": 3, "lingonberry": 5},
      "total":   20,
      "image_size": [width, height]
    }

Requirements:  pip install ultralytics onnxruntime pillow
"""
import io
from pathlib import Path

from PIL import Image
from ultralytics import YOLO

CLASSES = ["bilberry", "cloudberry", "crowberry", "lingonberry"]


class BerryDetector:
    def __init__(self, model_path="best.onnx", conf=0.25, imgsz=640):
        self.model = YOLO(str(model_path), task="detect")
        self.conf = conf
        self.imgsz = imgsz

    def _to_image(self, image):
        if isinstance(image, (bytes, bytearray)):
            return Image.open(io.BytesIO(image)).convert("RGB")
        if isinstance(image, (str, Path)):
            return Image.open(image).convert("RGB")
        return image  # PIL image or numpy array

    def detect(self, image, conf=None):
        img = self._to_image(image)
        res = self.model.predict(img, conf=conf or self.conf, imgsz=self.imgsz, verbose=False)[0]
        berries = []
        for box, c, p in zip(res.boxes.xyxy.tolist(), res.boxes.cls.tolist(), res.boxes.conf.tolist()):
            berries.append({"berry": CLASSES[int(c)], "confidence": round(float(p), 3),
                            "box": [round(v, 1) for v in box]})
        counts = {name: 0 for name in CLASSES}
        for b in berries:
            counts[b["berry"]] += 1
        h, w = res.orig_shape
        return {"berries": berries, "counts": counts, "total": len(berries), "image_size": [w, h]}


if __name__ == "__main__":
    import json, sys
    det = BerryDetector(sys.argv[2] if len(sys.argv) > 2 else "best.onnx")
    print(json.dumps(det.detect(sys.argv[1]), indent=2))


In [ ]:
import shutil, os, sys, glob
H = "/kaggle/working/handoff"
os.makedirs(H + "/examples", exist_ok=True)
shutil.copy("/kaggle/working/runs/wildbe_v1/weights/best.pt", H)
shutil.copy(onnx_path, H + "/best.onnx")
shutil.copy("/kaggle/working/predict.py", H)
json.dump(metrics, open(H + "/metrics.json", "w"), indent=2)
for f in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    p = f"/kaggle/working/runs/wildbe_v1/{f}"
    if os.path.exists(p): shutil.copy(p, H + "/examples/train_" + f)

# Check predict.py really works on the ONNX model, on 3 test images
sys.path.insert(0, H)
from predict import BerryDetector
det = BerryDetector(H + "/best.onnx")
test_imgs = sorted(glob.glob("/tmp/wildbe_yolo/images/test/*"))[:3]
for i, p in enumerate(test_imgs):
    out = det.detect(p)
    print(os.path.basename(p), out["counts"], "total:", out["total"])
    det.model.predict(p, imgsz=640, verbose=False)[0].save(filename=f"{H}/examples/test_{i+1}.jpg")

In [ ]:
readme = f"""# Berry detector – handoff for the dev team (Team Picky)

## What it does
Finds and counts **bilberry, cloudberry, crowberry and lingonberry** in a photo.
Used by the backend **Yield Prediction** service (berry counts feed the zone yield score).

## Files
| File | What it is |
|---|---|
| `best.onnx` | Trained model (YOLO11n, 640 px) – **use this in the backend** |
| `best.pt` | Same model in PyTorch format (for retraining) |
| `predict.py` | Ready-to-use `BerryDetector` class |
| `metrics.json` | Accuracy on unseen test images |
| `examples/` | Example detections and training curves |

## Install
```
pip install ultralytics onnxruntime pillow
```

## Use in FastAPI
```python
from fastapi import FastAPI, UploadFile
from predict import BerryDetector

app = FastAPI()
detector = BerryDetector("best.onnx")   # load once at startup

@app.post("/detect")
async def detect(file: UploadFile):
    return detector.detect(await file.read())
```

## Output
```json
{{"berries": [{{"berry": "bilberry", "confidence": 0.87, "box": [x1, y1, x2, y2]}}],
 "counts": {{"bilberry": 12, "cloudberry": 0, "crowberry": 3, "lingonberry": 5}},
 "total": 20, "image_size": [width, height]}}
```

## Accuracy (test set, never seen in training)
- mAP50: **{metrics["test_mAP50"]}** · mAP50-95: {metrics["test_mAP50_95"]} · precision {metrics["precision"]} · recall {metrics["recall"]}
- Per berry (mAP50): {", ".join(f"{k} {v}" for k, v in metrics["per_class_mAP50"].items())}

## Notes
- Default confidence threshold 0.25 (`BerryDetector(conf=0.4)` for fewer false alarms).
- Trained on WildBe drone + phone images from Finnish forests. Licence **CC BY-NC 4.0**: OK for the course/demo, **not for a paid product**.
- Citation: Riz et al. (2025), *Wild Berry Image Dataset Collected in Finnish Forests and Peatlands Using Drones*, ECCV 2024 Workshops.
"""
open(H + "/README.md", "w").write(readme)
shutil.make_archive("/kaggle/working/handoff", "zip", H)
!ls -la /kaggle/working/handoff /kaggle/working/handoff.zip

## ✅ Done
Download **`handoff.zip`** from the **Output** tab of this notebook version and give it to the dev team.